# B2-020 — Session 3: Pretraining Objectives

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).



## 1. Two pretraining objectives

Causal language modeling predicts the next token using only the prefix. Masked language modeling predicts selected hidden tokens using left and right context. Both are nlp-pretraining-objectives, but their visibility contracts and deployment matches differ.

**Checkpoint 1A.** Choose the objective for left-to-right generation.

**Checkpoint 1B.** Choose the objective for bidirectional encoding.

In [ ]:
import torch
torch.manual_seed(20260812)
ATOL = 1e-6
RTOL = 1e-6
assert torch.get_num_threads() >= 1

## 2. Masked-token corruption

Copy the literal row, replace every selected true token with `<mask>` ID 1, and set all unselected labels to `-100`. The model must never observe the selected true token in its input.

**Checkpoint 2A.** Why copy before corruption?

**Checkpoint 2B.** What does `-100` mean to cross-entropy?

## 3. Leakage counterexample

If the true token remains visible at a selected position, a high-capacity model can copy it without learning contextual prediction. This produces an impressive loss for the wrong task, so input/label disjointness is part of the objective definition.

**Checkpoint 3A.** Construct the one-token leak.

**Checkpoint 3B.** Why does a low leaked loss not certify pretraining?

## 4. Architecture delta: learned positions

The committed pretraining encoder replaces B2-019's sinusoidal table with an `nn.Embedding(8,8)` learned positional table. Positions 0–7 are parameters and receive gradients; sequence length remains at most 8.

**Checkpoint 4A.** Contrast fixed sinusoidal and learned positions.

**Checkpoint 4B.** Trace the broadcast shape.

## 5. Architecture delta: GELU and the pinned pretraining block

The feed-forward network uses `Linear(8,16)`, GELU, then `Linear(16,8)`. GELU is a smooth gate that scales inputs by a Gaussian-CDF-shaped factor; it replaces B2-019's ReLU in this pinned stack. The Session-3 p19 pretraining architecture is pre-norm: `LayerNorm(8, eps=1e-5)` precedes attention and the feed-forward sublayer, residual connections remain around both sublayers, and attention dropout is 0.0. Together with learned `nn.Embedding(8,8)` positions, these choices make one reproducible pretraining architecture rather than a family of compatible models. Session-2 p18 instead keeps B2-019's fixed sinusoidal positions, so it can be completed before this learned-position delta.

**Checkpoint 5A.** Trace the two linear shapes.

**Checkpoint 5B.** What stays unchanged around the feed-forward network?

## 6. AdamW and the sequential protocol

AdamW is Adam with decoupled weight decay. Here `weight_decay=0`, so it is numerically Adam. One AdamW instance with `lr=0.03`, betas `(0.9,0.999)`, eps `1e-8`, no AMSGrad/foreach/fused runs 40 causal updates, then continues its state through 40 MLM updates. Seeded reproducibility also includes construction order: after setting the seed, construct `TinyEncoder` before the distinct vocabulary head, because each constructor consumes random draws.

**Checkpoint 6A.** Why is a second optimizer incorrect?

**Checkpoint 6B.** What should optimizer step 41 mean?

## 7. Reproducible traces and selection

Every update consumes literal full batches in stored order, uses mean token cross-entropy, then zero-grad, backward, step. A phase trace records phase, local update index, mask mode, optimizer step, and loss. Choose an objective from the deployment visibility contract, not from fashion.

**Checkpoint 7A.** List the five trace fields.

**Checkpoint 7B.** What mask mode belongs to MLM?

## 8. Common pitfalls, Exam connections, and Going deeper

Pitfalls include causal masking during MLM, leaving targets visible, resetting AdamW, and calling a random state pretrained. Exams ask objective selection and leakage repair. Going deeper names span corruption and replaced-token detection without using them before they are taught.

**Checkpoint 8A.** Which mutation repeats optimizer steps 1–40?

**Checkpoint 8B.** Which practice runs both phases?

## Collected checkpoint answers

**Answer 1A.** causal LM for left-to-right generation.  **Answer 1B.** MLM for bidirectional encoding.

**Answer 2A.** copying preserves true labels before inputs are changed.  **Answer 2B.** `-100` excludes an unselected position from cross-entropy.

**Answer 3A.** leave a selected true token visible and predict that same token.  **Answer 3B.** copying the visible target can lower loss without contextual learning.

**Answer 4A.** sinusoidal positions are fixed; learned positions are trainable `Embedding(8,8)` rows.  **Answer 4B.** `(8,)` positions broadcast to `(B,8,8)`.

**Answer 5A.** `8 -> 16 -> 8`.  **Answer 5B.** residual structure and width 8 stay unchanged.

**Answer 6A.** it resets AdamW moments and step state.  **Answer 6B.** it is the first MLM update with optimizer step 41.

**Answer 7A.** phase, local update index, mask mode, optimizer step, loss.  **Answer 7B.** bidirectional.

**Answer 8A.** resetting AdamW repeats steps 1--40.  **Answer 8B.** p19.